In [ ]:
import random
from datasets import load_dataset, get_dataset_config_names
import pandas as pd

# ==============================================================================
# 🛡️ 데이터셋 정보 및 목표 설명 (White Hat Security Agent Prompts 600K)
# ==============================================================================
# 이 데이터셋은 '화이트 햇 보안 에이전트의 프롬프트 모음'을 목표로 합니다.
# 즉, 실제 보안 전문가들이 위협(Threat)을 분석하거나, 방어 전략을 세우거나,
# 취약점을 진단할 때 사용하는 '질문(user_prompt)'들의 방대한 코퍼스입니다.
#
# 🔥 이 실습의 목표: 단순히 데이터를 보여주는 것이 아니라, AI 모델이 이 데이터를
# 마치 실제 업무 지침처럼 '활용'하여 위협을 분석하고 방어 계획을 짜는 과정을
# 시뮬레이션해보는 '프롬프트 엔지니어링' 기획 연습입니다.
#
# 🔬 특징:
# - user_prompt: 보안 전문가가 던지는 질문이나 시나리오 (핵심!)
# - threat: 발견된 구체적인 위협 유형 (예: Malware, DDoS)
# - impact_level: 위협이 미치는 피해의 심각도 (High, Medium, Low)
# ==============================================================================

# --- 설정값 ---
DATASET_NAME = "yatin-superintelligence/White-Hat-Security-Agent-Prompts-600K"
SAMPLE_COUNT = 5 # 초보자 테스트를 위해 샘플 수를 매우 작게 설정합니다!

# ==============================================================================
# 🔧 1. 데이터셋 로드 및 설정
# ==============================================================================

print("🌟 [튜터의 인사] 안녕하세요! 오늘 함께 방어 전문가(White Hat)의 언어를 학습해볼 거예요!")
print("🛡️ 이 데이터셋은 수십만 개의 '보안 질문'들로 가득 찬 보물창고와 같아요.")
print("💡 우리는 데이터 분석을 통해 AI 에이전트가 어떻게 생각하는지 관찰할 겁니다.")

try:
    # 1단계: 데이터셋 Config 목록 확인 (필수 과정!)
    configs = get_dataset_config_names(DATASET_NAME)
    print(f"\n✅ 사용 가능한 Config 목록: {configs}")
    
    # 가장 기본적인 'default' 설정을 선택합니다.
    selected_config_name = configs[0] 

    # 2단계: 스트리밍 방식으로 데이터셋 로드 시도 (가장 빠르고 메모리 효율적!)
    print("\n🚀 [Step 1/3] 스트리밍(Streaming) 모드로 데이터셋을 로드하는 중...")
    dataset = load_dataset(DATASET_NAME, name=selected_config_name, split='train', streaming=True)

except Exception as e:
    # 스트리밍 로드 실패 시 (네트워크 또는 버전 문제 등), 일반 모드로 전환
    print(f"\n⚠️ 스트리밍 로드 실패 ({e}). 일반 로드 모드(streaming=False)로 전환합니다.")
    try:
        dataset = load_dataset(DATASET_NAME, name=selected_config_name, split='train')
    except Exception as e_fallback:
        print(f"🚨 치명적 오류: 데이터셋을 로드할 수 없습니다. 인터넷 연결을 확인해주세요. ({e_fallback})")
        dataset = None

# ==============================================================================
# 🔄 2. 샘플 데이터셋 확보 (테스트 환경 최적화)
# ==============================================================================

if dataset is None:
    exit() # 데이터 로드 실패 시 프로그램 종료

# 데이터셋의 크기를 알 수 없으므로 (Streaming 모드일 경우),
# .take() 메서드를 사용하여 상위 K개만 메모리에 올리겠습니다.
print(f"\n✨ [Step 2/3] 상위 {SAMPLE_COUNT}개의 샘플 데이터만 추출하여 작업 환경을 설정합니다.")

# Streaming 모드 체크 및 데이터셋 처리
if hasattr(dataset, "take"):
    # 스트리밍 데이터셋 (IterableDataset)인 경우
    sampled_dataset_iterator = dataset.take(SAMPLE_COUNT)
    # Iterator를 리스트로 변환하여 반복 가능하게 만듭니다.
    sample_data_list = list(sampled_dataset_iterator)
else:
    # 일반 Dataset인 경우
    sample_data_list = list(dataset.select(range(SAMPLE_COUNT)))


print(f"✅ 성공! 총 {len(sample_data_list)}개의 샘플 데이터({SAMPLE_COUNT}개)를 확보했습니다. 이제 실습에 들어갑니다!")


# ==============================================================================
# 📊 3. 실습 1: 데이터 분포 분석 (Impact Level 분석)
# ==============================================================================
print("\n\n=======================================================")
print("🎯 [실습 1] 데이터 분포 분석: 이 위협들은 얼마나 심각할까요?")
print("=======================================================")

impact_counts = {}
for sample in sample_data_list:
    impact = sample.get('impact_level', 'N/A')
    impact_counts[impact] = impact_counts.get(impact, 0) + 1

# 결과 출력 (정량적 분석 포함)
print("📊 [Impact Level] 분석 결과:")
for level, count in impact_counts.items():
    print(f"  - {level} 레벨 위협: {count} 건 (가장 위험한 위협은 이 레벨일 가능성이 높습니다!)")

# ==============================================================================
# 🧠 4. 실습 2: AI 에이전트 시뮬레이션 (가장 창의적인 부분!)
# ==============================================================================
print("\n\n=======================================================")
print("🤖 [실습 2] AI 에이전트 모드 시뮬레이션: 위협 분석가 되기!")
print("=======================================================")
print("🤔 우리는 AI가 보안 프롬프트 하나를 받았을 때, 어떻게 구조화하여 답변할지 시뮬레이션 해볼 겁니다.")

# 샘플 3개를 무작위로 선택하여 테스트합니다. (전체 샘플 중 무작위 선택)
if len(sample_data_list) < 3:
    test_samples = sample_data_list
else:
    test_samples = random.sample(sample_data_list, 3)

for i, sample in enumerate(test_samples):
    print(f"\n================= [Case Study #{i+1}] ==================")
    
    user_prompt = sample.get('user_prompt', 'N/A')
    threat = sample.get('threat', 'Unknown')
    impact_level = sample.get('impact_level', 'Unknown')

    print(f"🔍 [수신된 프롬프트]: \"{user_prompt[:70]}...\" (실제 질문 내용)")
    print(f"   ➡️ [분석된 위협]: {threat} | [위험도]: {impact_level}")

    # 🧠 AI 에이전트의 답변 구조화 시뮬레이션
    print("\n✨ [AI Security Agent의 분석 보고서 초안]")
    
    # 1. 위험도에 따른 경고 메시지 생성
    if impact_level == "High":
        print("🚨 [긴급 경고] 이 위협은 매우 심각합니다! 즉각적인 조치가 필요합니다.")
        advice = "즉시 패치 적용, 네트워크 격리, 침입 경로 추적을 최우선으로 수행해야 합니다."
    elif impact_level == "Medium":
        print("⚠️ [경고] 주의가 필요하며, 점진적인 대응이 필요합니다.")
        advice = "취약점 분석 및 접근 통제 강화 등의 중간 단계적 조치가 요구됩니다."
    else:
        print("✅ [상태 확인] 위협은 확인되었으나, 당장 최대의 위협은 아닙니다.")
        advice = "정기적인 모니터링을 통해 패턴을 분석하고 예방책을 마련하는 것이 좋습니다."

    print(f"   -> [핵심 분석]: 발견된 {threat}에 대한 깊이 있는 분석을 시작합니다.")
    print(f"   -> [최적 방어 조치]: {advice}")
    print("-------------------------------------------------------")

print("\n\n🎉✨ [튜터 코멘트] 정말 잘하셨어요!!")
print("이제 여러분은 단순한 데이터 사용자를 넘어, AI가 '어떤 구조와 논리'로 사고해야 하는지 이해하는 '프롬프트 설계자'가 된 거예요.")
print("이처럼 데이터를 분석하고, 그 결과에 따라 논리적인 텍스트를 생성하는 과정이 바로 'AI 애플리케이션'의 핵심입니다!")